# Week 39, Day 6: Exam debrief

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w039d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


In [ ]:
class Interval:
    def __init__(self, lo, hi):
        if lo > hi:
            raise ValueError(f"lo must not be above hi: {lo} > {hi}")
        if lo == hi:
            lo = hi = 0
        self.lo, self.hi = lo, hi

    def __repr__(self):
        return f"Interval({self.lo}, {self.hi})"

    def __len__(self):
        return self.hi - self.lo

    def __contains__(self, n):
        return self.lo <= n < self.hi

    def __iter__(self):
        return iter(range(self.lo, self.hi))

    def __eq__(self, other):
        if not isinstance(other, Interval):
            return NotImplemented
        return (self.lo, self.hi) == (other.lo, other.hi)

    def __hash__(self):
        return hash((self.lo, self.hi))

    def __and__(self, other):
        if not isinstance(other, Interval):
            return NotImplemented
        lo, hi = max(self.lo, other.lo), min(self.hi, other.hi)
        return Interval(lo, hi) if lo < hi else Interval(0, 0)
a, b = Interval(2, 5), Interval(4, 9)
print(a & b, list(a), 4 in a, len(Interval(0, 3) & Interval(3, 6)), Interval(3, 3) == Interval(0, 0))

In [ ]:
from functools import total_ordering

class CurrencyError(ValueError):
    pass

@total_ordering
class Money:
    def __init__(self, pence, currency):
        self.pence = pence
        self.currency = currency.upper()

    def _same(self, other):
        if self.currency != other.currency:
            raise CurrencyError(f"can't mix {self.currency} and {other.currency}")

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._same(other)
        return Money(self.pence + other.pence, self.currency)

    def __sub__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._same(other)
        return Money(self.pence - other.pence, self.currency)

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.pence, self.currency) == (other.pence, other.currency)

    def __hash__(self):
        return hash((self.pence, self.currency))

    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._same(other)
        return self.pence < other.pence

    def __str__(self):
        sign = "-" if self.pence < 0 else ""
        pounds, rest = divmod(abs(self.pence), 100)
        return f"{self.currency} {sign}{pounds}.{rest:02d}"
print(Money(1250, "gbp") + Money(75, "GBP"), Money(1, "GBP") == Money(1, "NGN"))
try:
    Money(1, "GBP") < Money(1, "NGN")
except CurrencyError as err:
    print("CurrencyError:", err)

In [ ]:
class BadMove(ValueError):
    pass

class Order:
    MOVES = {
        ("placed", "pay"): "paid",
        ("placed", "cancel"): "cancelled",
        ("paid", "ship"): "shipped",
        ("paid", "cancel"): "refunded",
        ("shipped", "deliver"): "delivered",
    }

    def __init__(self):
        self.state = "placed"
        self.history = []

    def handle(self, event):
        new = self.MOVES.get((self.state, event))
        if new is None:
            raise BadMove(f"can't {event} a {self.state} order")
        self.history.append(self.state)
        self.state = new
        return new

    def undo(self):
        if not self.history:
            return False
        self.state = self.history.pop()
        return True
o = Order()
print(o.handle("pay"), o.handle("ship"), o.history, o.undo(), o.state)
try:
    o.handle("deliver")
except BadMove as err:
    print(err)

In [ ]:
class Feed:
    def __init__(self):
        self._posts = []
        self._listeners = []

    def subscribe(self, listener):
        self._listeners.append(listener)

    def unsubscribe(self, listener):
        self._listeners.remove(listener)

    def post(self, text):
        if not text.strip():
            raise ValueError("a post can't be blank")
        self._posts.append(text)
        for listener in list(self._listeners):
            listener(text)

    def __iter__(self):
        return iter(self._posts[::-1])

    def __len__(self):
        return len(self._posts)
f = Feed()
f.subscribe(lambda text: print("new post:", text))
f.post("hello")
f.post("again")
print(list(f), list(f), len(f))

In [ ]:
from typing import Callable

def group_by[T, K](items: list[T], key: Callable[[T], K]) -> dict[K, list[T]]:
    groups: dict[K, list[T]] = {}
    for item in items:
        groups.setdefault(key(item), []).append(item)
    return groups

def first_where[T](items: list[T], test: Callable[[T], bool]) -> T | None:
    for item in items:
        if test(item):
            return item
    return None
print(group_by(["apple", "fig", "avocado"], lambda w: w[0]), first_where([1, 4, 6], lambda n: n % 2 == 0))
print(group_by.__annotations__)

In [ ]:
class Team:
    def __init__(self, name):
        self.name = name
        self.members = []

    def add(self, person):
        if person in self.members:
            raise ValueError(f"{person} is already in {self.name}")
        self.members.append(person)

    def __eq__(self, other):
        if not isinstance(other, Team):
            return NotImplemented
        return self.name == other.name

    def __hash__(self):
        return hash(self.name)

class Squad(Team):
    def __init__(self, name, coach):
        super().__init__(name)
        self.coach = coach

    def describe(self):
        return f"{self.name} (coach {self.coach}): {', '.join(self.members)}"
a, b = Team("Lions"), Team("Tigers")
a.add("Ada")
s = Squad("Hawks", "Bo")
s.add("Chi")
print(b.members, len({Team("Lions"), a}), s.describe())

In [ ]:
from dataclasses import dataclass


class LeagueError(Exception):
    pass


class UnknownTeam(LeagueError, LookupError):
    pass


class BadResult(LeagueError, ValueError):
    pass


@dataclass(frozen=True)
class Result:
    home: str
    away: str
    home_goals: int
    away_goals: int

    def __str__(self):
        return f"{self.home} {self.home_goals}-{self.away_goals} {self.away}"


def _is_score(word):
    left, dash, right = word.partition("-")
    return bool(dash) and all(p.isascii() and p.isdigit() for p in (left, right))


def parse_result(line):
    words = line.split()
    scores = [i for i, w in enumerate(words) if _is_score(w)]
    if len(scores) != 1:
        raise BadResult(f"bad result: {line!r}")
    i = scores[0]
    home, away = " ".join(words[:i]), " ".join(words[i + 1:])
    if not home or not away or home == away:
        raise BadResult(f"bad result: {line!r}")
    left, _, right = words[i].partition("-")
    return Result(home, away, int(left), int(right))


def standard_points(goals_for, goals_against):
    if goals_for > goals_against:
        return 3
    return 1 if goals_for == goals_against else 0


class League:
    def __init__(self, teams, points=standard_points):
        self.teams = list(teams)
        self.points = points
        self.results = []

    def record(self, line):
        result = parse_result(line)
        for name in (result.home, result.away):
            if name not in self.teams:
                raise UnknownTeam(f"no team {name!r}")
        self.results.append(result)
        return result

    def undo(self):
        if not self.results:
            raise LeagueError("nothing to undo")
        return self.results.pop()

    def table(self):
        rows = {t: [0, 0, 0, 0, 0, 0, 0] for t in self.teams}    # played, won, drawn, lost, for, against, points
        for r in self.results:
            for team, scored, conceded in ((r.home, r.home_goals, r.away_goals), (r.away, r.away_goals, r.home_goals)):
                row = rows[team]
                row[0] += 1
                row[1 if scored > conceded else 2 if scored == conceded else 3] += 1
                row[4] += scored
                row[5] += conceded
                row[6] += self.points(scored, conceded)
        table = [(team, *row) for team, row in rows.items()]
        return sorted(table, key=lambda t: (-t[7], -(t[5] - t[6]), -t[5], t[0]))


def handle(line, league):
    try:
        command = line.strip()
        if command == "table":
            return "\n".join(f"{i}. {t} P{p} W{w} D{d} L{l} F{f} A{a} Pts{pts}"
                             for i, (t, p, w, d, l, f, a, pts) in enumerate(league.table(), start=1))
        if command == "undo":
            return f"removed {league.undo()}"
        return f"recorded {league.record(line)}"
    except LeagueError as err:
        return f"error: {err}"
league = League(["Red Lions", "Blue Tigers", "Owls", "Green Bees"])
for line in ["Red Lions 3-1 Blue Tigers", "Owls 2-2 Red Lions", "Blue Tigers 1-0 Owls", "Owls 1-1 Owls", "Bats 2-0 Owls", "table", "undo"]:
    print(f"> {line}")
    print(handle(line, league))

### Exercise: Second chance: version numbers

Write `Version(text)` for versions like `"1.10.2"` (one or more whole numbers separated by dots; anything else raises `ValueError`). Versions compare by their numbers, so `1.10 > 1.9`, and trailing zeros don't matter: `Version("1.2") == Version("1.2.0")`. Give it `__eq__`, `__hash__` (consistent with that), `__lt__` with `total_ordering`, `__str__` (the text as given, stripped) and `__repr__` (`Version('1.10.2')`).

In [ ]:
from functools import total_ordering

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_versions():\n    \"versions\"\n    assert Version(\"1.10\") > Version(\"1.9\") and Version(\"1.2\") == Version(\"1.2.0\") and len({Version(\"2\"), Version(\"2.0.0\")}) == 1\n    assert sorted(Version(v) for v in [\"1.10\", \"1.2\", \"1.9.1\", \"0.9\"]) == [Version(\"0.9\"), Version(\"1.2\"), Version(\"1.9.1\"), Version(\"1.10\")]\n    assert str(Version(\" 1.10.2 \")) == \"1.10.2\" and repr(Version(\"1.10.2\")) == \"Version('1.10.2')\"\n    for bad in [\"\", \"1..2\", \"1.x\", \"v1.2\", \"1.2.\"]:\n        try:\n            Version(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: Second chance: a deck

Write `Deck(cards)`, a container holding a copy of the list of cards: `len`, indexing (`deck[0]` is the top card), `in`, iteration from the top, and `draw(n=1)`, which removes and returns a list of the top `n` cards (asking for more than there are raises `ValueError` and draws nothing). Iterating must work any number of times.

In [ ]:
class Deck:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deck():\n    \"a deck\"\n    cards = [\"AS\", \"KH\", \"7C\", \"2D\"]\n    d = Deck(cards)\n    cards.append(\"XX\")\n    assert len(d) == 4 and d[0] == \"AS\" and d[-1] == \"2D\" and \"7C\" in d and \"XX\" not in d\n    assert list(d) == [\"AS\", \"KH\", \"7C\", \"2D\"] and list(d) == list(d)\n    assert d.draw(2) == [\"AS\", \"KH\"] and d.draw() == [\"7C\"] and len(d) == 1\n    try:\n        d.draw(2)\n        assert False\n    except ValueError:\n        assert len(d) == 1")

### Exercise: Second chance: shifts

Write `Day`, an Enum of `MON` to `SUN` with the values `"mon"` to `"sun"`; `Shift`, a frozen, ordered dataclass of `day` (a `Day`), `start` and `end` (hours, ints) that sorts by the day's position in the week, then start, with `__post_init__` refusing `end <= start` or hours outside 0 to 24 (`ValueError`); and `parse_shift(text)` for `"Mon 9-17"` (any case). Because Enums don't sort, store a hidden sort field: `order: int = field(init=False, repr=False)` set in `__post_init__` with `object.__setattr__` (it's frozen), and give `day` `compare=False`.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shifts():\n    \"shifts\"\n    s = parse_shift(\"Mon 9-17\")\n    assert s.day is Day.MON and (s.start, s.end) == (9, 17)\n    got = sorted([parse_shift(\"sun 8-12\"), parse_shift(\"TUE 13-20\"), parse_shift(\"tue 6-10\"), parse_shift(\"mon 9-17\")])\n    assert [(x.day.value, x.start) for x in got] == [(\"mon\", 9), (\"tue\", 6), (\"tue\", 13), (\"sun\", 8)]\n    for bad in [\"mon 17-9\", \"fri 20-25\", \"xyz 9-17\"]:\n        try:\n            parse_shift(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: Second chance: a ticket gate

Write `Gate`, a state machine with a transition table: `"locked"` + `ticket` → `"open"`; `"open"` + `push` → `"locked"` (and the count of people through goes up by 1); `"locked"` + `push` → `"alarm"`; `"alarm"` + `reset` → `"locked"`. Anything else raises `ValueError("can't EVENT when STATE")`. `handle(event)` returns the new state; `through` is the count.

In [ ]:
class Gate:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gate():\n    \"a gate\"\n    g = Gate()\n    assert [g.handle(e) for e in [\"ticket\", \"push\", \"push\", \"reset\", \"ticket\", \"push\"]] == [\"open\", \"locked\", \"alarm\", \"locked\", \"open\", \"locked\"]\n    assert g.through == 2\n    try:\n        g.handle(\"reset\")\n        assert False\n    except ValueError as err:\n        assert str(err) == \"can't reset when locked\"")

### Bug hunt: Second chance: three more bugs

`Playlist` has three bugs: a mutable default argument shared between playlists, an `__eq__` that crashes when compared with something that isn't a playlist (it should return `NotImplemented`), and a subclass that never calls the parent's `__init__`. Fix them.

In [ ]:
class Playlist:
    def __init__(self, name, songs=[]):
        self.name = name
        self.songs = songs

    def add(self, song):
        self.songs.append(song)

    def __eq__(self, other):
        return self.songs == other.songs

    def __hash__(self):
        return hash(tuple(self.songs))

class Radio(Playlist):
    def __init__(self, name, station):
        self.station = station

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"fixed\"\n    a, b = Playlist(\"a\"), Playlist(\"b\")\n    a.add(\"Zombie\")\n    assert b.songs == [] and (a == \"Zombie\") is False\n    r = Radio(\"Morning\", \"FM 99\")\n    r.add(\"Lady\")\n    assert r.name == \"Morning\" and r.songs == [\"Lady\"] and a.songs == [\"Zombie\"]")

### Exercise: Second chance: tests that bite

`clamp(n, lo, hi)` in `limits.py` returns `n` limited to the range `lo` to `hi` (both included), and raises `ValueError` if `lo > hi`. Write pytest tests that catch all four broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('limits.py', 'w', encoding='utf-8') as f:
    f.write('def clamp(n, lo, hi):\n    if lo > hi:\n        raise ValueError("lo is above hi")\n    return max(lo, min(n, hi))\n')

In [ ]:
import pytest
from limits import clamp

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"limits.py\").read()\nBROKEN = {\n    \"lo and hi swapped\": GOOD.replace(\"return max(lo, min(n, hi))\", \"return min(lo, max(n, hi))\"),\n    \"equal bounds refused\": GOOD.replace(\"if lo > hi:\", \"if lo >= hi:\"),\n    \"no check\": GOOD.replace('    if lo > hi:\\n        raise ValueError(\"lo is above hi\")\\n', \"\"),\n    \"top ignored\": GOOD.replace(\"return max(lo, min(n, hi))\", \"return max(lo, n)\"),\n}\ndef test_catches():\n    \"your tests pass on the real code and catch all four broken versions\"\n    results = against(source, \"limits\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"limits.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Your score

Write down your total before reading on. Then, as you go through each problem below, note **why** you lost any marks. There are only a few kinds of reason, and knowing which kind you make most is more useful than the score itself:

1. **Didn't know it**: a gap in what you've learned. Revise the topic.
2. **Misread the question**: the code did something sensible but different. Slow down; underline every requirement.
3. **Edge case**: the main case worked, but empty input, a tie or a boundary failed. Test small and extreme cases.
4. **Ran out of time**: the plan, not the knowledge, was the problem. Practise starting with the parts you're sure of.

| Score | What it means |
|---|---|
| 144 or more (80%) | Distinction. You're well prepared for Phase 4. |
| 108 to 143 (60%) | Pass. Revise the topics where you lost marks during the rest week. |
| under 108 | Not yet. Spend the rest week on the topics below, then take the exam again. What matters is the second attempt. |

## Part A

| Questions | The idea | Revise |
|---|---|---|
| 1 | a mutable class attribute is shared by every instance | [Week 28, Day 5](w028d5.html) |
| 2, 7 | `self` finds the most specific method; `super()` follows the object's MRO | [Week 30, Day 1](w030d1.html), [Week 30, Day 3](w030d3.html) |
| 3, 5, 6 | `__repr__` and `__str__`; `__eq__` with `__hash__`; `total_ordering` | [Week 29, Days 1 and 2](w029d1.html), [Week 31, Day 3](w031d3.html) |
| 4 | the container protocols | [Week 31, Day 2](w031d2.html) |
| 8 | composition versus inheritance | [Week 30, Day 5](w030d5.html) |
| 9, 10, 11 | frozen dataclasses; Enums by value and by name; ABCs against Protocols | [Week 32](w032d1.html) |
| 12 | `X \| None` and checking for `None` | [Week 33, Day 2](w033d2.html) |
| 13 | single responsibility | [Week 34, Day 2](w034d2.html) |
| 14, 15, 16 | catching by family and `raise from`; who configures logging; `assert` against `raise` | [Week 37](w037d1.html) |
| 17, 18 | Observer; iterables against iterators | [Week 35, Day 2](w035d2.html), [Week 36, Day 3](w036d3.html) |
| 19, 20 | the dependency rule; injecting a clock | [Week 36, Days 4 and 5](w036d4.html) |

Question 7 catches many people: `super()` inside `Y` calls `Z`, not `X`, because it follows the MRO of the **object** (`W, Y, Z, X, object`), not `Y`'s own parents.

## Part B

Each model solution runs, so try it. The notes under each one explain the key decision and the most common mistakes.

### B1: Intervals

Normalising every empty interval to `Interval(0, 0)` in `__init__` is the trick that makes "all empty intervals are equal" (and hash equally) without any special cases in `__eq__` or `__hash__`.

**Common mistakes:** `__contains__` that includes `hi`; an overlap computed with `min` and `max` the wrong way round; forgetting `__hash__` after writing `__eq__`.

### B2: Money in currencies

One helper, `_same`, holds the currency rule, so `+`, `-` and `<` can't disagree about it (DRY). `__eq__` deliberately doesn't use it: asking whether £1 equals ₦1 has a sensible answer (no), but asking which is smaller doesn't.

**Common mistakes:** upper-casing the currency only when comparing; `str` of a negative amount written as `GBP 0.-5` (take the sign off first, then `divmod` the absolute value); raising for `==` between currencies.

### B3: Predict

`a` is `[1, 2]` because `super().__init__()` runs `Base.__init__`, which counts; `b` is 2. Jobs sort by `priority` only (`name` has `compare=False`), and `sorted` is stable, so the two priority-2 jobs keep their original order: `["z", "x", "a"]`. For the same reason `d` is `True`. The MRO of `Child` is `Child, Base, object`.

**Common mistake:** thinking `compare=False` fields break ties. They don't take part at all.

### B4: An order's life

Undo here is the simplest possible Command history: since every move just changes the state, remembering the previous **state** is enough. When commands do more (Week 36's editor), each command object remembers how to undo itself.

**Common mistakes:** putting the current state in `history`; an `if` chain instead of a table; `undo` that raises instead of returning `False`.

### B5: A feed

`self._posts[::-1]` makes a **new** reversed list each time `__iter__` is called. That one expression gives both requirements: a fresh iterator for every loop, and a snapshot that posts added during the loop can't disturb.

**Common mistakes:** `__iter__` returning `self` (only one loop works); `reversed(self._posts)`, which is a live view of the list (a post added mid-loop can make it misbehave); notifying observers from a list that a listener might change.

### B6: Typed helpers

The type parameters connect the hints: `group_by` returns lists of the **same** `T` it was given, and keys of whatever type `key` returns. A caller who groups `list[str]` by `len` gets `dict[int, list[str]]`, and mypy knows it.

**Common mistakes:** `Callable[T, K]` (the arguments go in a list: `Callable[[T], K]`); returning `T` instead of `T | None` from `first_where`.

### B7: Three bugs

The three bugs were Week 28's shared class attribute (move `members` into `__init__`), Week 29's unhashable class (add `__hash__` consistent with `__eq__`), and Week 30's missing `super().__init__` (so `name` is set).

### B8: Test a password policy

A model answer:

```python
import pytest
from policy import check

def test_good():
    assert check("Abcdefghi1") == []

def test_nine_is_short():
    assert check("Abcdefgh1") == ["too short"]

@pytest.mark.parametrize("password, problem", [
    ("Abcdefghijk", "no digit"),
    ("abcdefghi12", "no capital"),
    ("Abcde fghi1", "contains spaces"),
])
def test_each(password, problem):
    assert check(password) == [problem]

def test_all_problems():
    assert check("a b") == ["too short", "no digit", "no capital", "contains spaces"]
```

`"Abcdefghi1"` is exactly 10 characters and passes everything: it catches both length mistakes when paired with the 9-character `"Abcdefgh1"`. Each parametrized case breaks **exactly one** rule, so it proves that rule is checked (the "sharp example" idea from Week 24). `test_all_problems` catches the version that reports only the first problem.

**Common mistakes:** examples that break several rules at once, so a missing check goes unnoticed; no example at exactly 9 and 10 characters; a capital-letter test that uses only digits and lower case, which can't tell `isupper` from `isalpha`.

## Part C

The decisions that matter:

- **The score word decides the split.** Team names have spaces, so the parser looks for the one word shaped like `3-1` and treats everything before and after it as the teams. "Exactly one such word" also rejects `Owls 1-0 2-1 Bees`.
- **The table is built fresh** from the list of results each time, so `undo` is just `pop`: nothing else needs to be kept in step. (Keeping running totals would make undo harder and is a classic source of bugs.)
- **One tuple key does the tie-breaks**: `(-points, -goal difference, -goals for, name)`: negating the numbers sorts them downwards while names still sort A to Z.
- **Points are a strategy**, so a league that gives 2 for a win needs no change to `League`.
- **The interface catches only `LeagueError`.**

**Common mistakes:** `line.split()` into exactly three parts (breaks on team names with spaces); `reverse=True` on the whole key (sorts names Z to A); leaving teams with no games out of the table; checking the away team before the home team.

## Second chances

A fresh problem for each skill. If you lost marks on the matching exam problem, do this one now, without looking at the solution above.

## Quiz

## What's next

**Your revision plan.** For each problem where you lost marks, write one line: the problem, the kind of mistake (from the list at the top), and the day you'll revisit. Do those days during the rest week. If you're retaking the exam, do it after the rest week, without looking at this page again first.

**Phase 4: Data structures and algorithms.** After a week's rest, Phase 4 opens the box. You've used lists, dictionaries and sets for 40 weeks; now you'll build them, and their relatives, yourself: dynamic arrays and why appending is fast on average, linked lists, stacks and queues, hash tables, trees, heaps and graphs, and the algorithm families that use them. The classes, special methods and tests from Phase 3 are exactly the tools you'll build them with.

You've finished Phase 3. Thirteen weeks ago you asked why objects exist at all; since then you've written classes that behave like Python's own types, described data with dataclasses, Enums and Protocols, checked types with mypy, learned the principles and patterns that keep programs understandable, made programs robust, and built a layered library system from an empty file. Take the rest week: you've earned it.